### India Open Buildings Conversion Script

This notebook converts India Open Buildings data — available at [https://gobs.aeee.in/downloads](https://gobs.aeee.in/downloads) — from large compressed CSV files (`.csv.gz`) into spatial formats such as GeoPackage (`.gpkg`) or GeoJSON.  
Each file contains building footprint polygons in Well-Known Text (WKT) format, along with attributes like area, height, floors, and land use.  
The script reads each CSV in memory-efficient chunks, repairs invalid geometries, and writes them incrementally to per-file GeoPackages suitable for GIS analysis.

**Contact:**</br>
Benny Istanto, GOST/DEC Data Group/The World Bank, bistanto@worldbank.org

In [1]:
# !pip -q install pandas fiona shapely pyproj  # ← uncomment if needed
# If GDAL Python bindings are unavailable, we’ll fall back to ogrinfo CLI for CreateSpatialIndex.
# We also detect pre-existing R*Tree via sqlite3 to avoid false warnings.

# =============================================================================
# csv_buildings_to_geopkg  (Notebook Single-Cell: fast + robust + index detection & creation)
# =============================================================================
"""
Stream-convert India Open Buildings CSV/.csv.gz files into per-file GeoPackages
(or GeoJSON), parsing WKT polygons from the 'geometry' column.

Why this notebook cell?
- Many India Open Buildings CSVs are huge (GB-scale). This code reads in chunks
  and writes features incrementally so memory stays bounded.
- Uses Fiona directly for streaming writes (no need to hold an entire GeoDataFrame).
- Repairs invalid polygons (self-intersections, rings) only when needed.
- Logs progress, errors, and per-file statistics.

Speed features:
- Vectorized WKT parsing via shapely.from_wkt (Shapely ≥2.0).
- Batch writes with sink.writerecords(...) to minimize Python↔GDAL overhead.
- Disables spatial index during write for speed & to avoid rtree UNIQUE errors,
  then creates the spatial index immediately afterwards (in this process), unless it already exists.

Robustness:
- Existing-output policy: "auto" | "replace" | "skip" | "abort".
- Schema declared as MultiPolygon and Polygons are promoted to MultiPolygon to avoid type errors.
- Spatial index builder first checks if the R*Tree already exists (sqlite3), then tries GDAL Python or ogrinfo.

Requirements:
  - Python 3.9+
  - pandas
  - fiona
  - shapely >= 2.0 preferred (for from_wkt & make_valid). If <2.0, fallbacks are used.
"""

from __future__ import annotations

import os
import sys
import logging
import shutil
import subprocess
import sqlite3
from glob import glob
from typing import Dict, Iterable, List, Optional, Tuple

import pandas as pd
import fiona
from fiona.crs import CRS
from shapely import wkt as shapely_wkt
from shapely.geometry import mapping, MultiPolygon  # ← MultiPolygon for schema + casting

# Prefer Shapely ≥2.0 fast-paths: from_wkt + make_valid
try:
    from shapely import from_wkt as shapely_from_wkt
    SHAPELY_HAS_FROM_WKT = True
except Exception:
    SHAPELY_HAS_FROM_WKT = False

try:
    from shapely.validation import make_valid as shapely_make_valid
    SHAPELY_HAS_MAKE_VALID = True
except Exception:
    SHAPELY_HAS_MAKE_VALID = False

# Try GDAL Python bindings for in-process spatial index creation
try:
    from osgeo import ogr  # type: ignore
    HAS_GDAL_PY = True
except Exception:
    HAS_GDAL_PY = False


# ============================== USER PARAMETERS ============================== #
INPUT_DIR = "/mnt/c/Users/benny/OneDrive/Personal/wbg/data/ind/gobs-csv"
OUTPUT_DIR = "/mnt/c/Users/benny/OneDrive/Personal/wbg/data/ind/gobs-gpkg"
PATTERN = "*.csv.gz"             # or "*.csv"
OUTPUT_FORMAT = "gpkg"           # "gpkg" or "geojson"
GPKG_LAYER = "buildings"
GEOM_NAME = "geom"               # GeoPackage geometry column name (fixed for deterministic SQL)
GEOM_SCHEMA_TYPE = "MultiPolygon"  # Layer schema geometry; Polygon rows will be promoted to MultiPolygon

CHUNK_SIZE = 250_000             # speed vs RAM (try 200k–500k)
WRITE_BATCH_SIZE = 25_000        # batch size for sink.writerecords; adaptive backoff on failure
CRS_EPSG = 4326                  # WGS84
GEOM_COL = "geometry"            # CSV WKT column
SELECTED_COLUMNS = None          # e.g. ["latitude","longitude","area_in_meters","landuse","state_name","district_name","geometry"]

# During write, disable spatial index for speed & to avoid rtree UNIQUE errors; build it right after
WRITE_WITH_SPATIAL_INDEX = False
BUILD_SPATIAL_INDEX_AFTER = True

# Existing-output behavior: "auto" | "replace" | "skip" | "abort"
EXISTING_BEHAVIOR = "skip"
ALLOW_SKIPS = 1000               # tolerance for 'auto' completeness check

# Geometry repair
REPAIR_INVALID = True

# Logging
VERBOSITY = 1                    # 0=WARNING, 1=INFO, 2=DEBUG
FID64 = True                     # safer on huge datasets


# ============================== Logging setup ================================ #
def setup_logger(verbosity: int = 1) -> logging.Logger:
    """
    Configure and return a module-level logger.

    Parameters
    ----------
    verbosity : int
        0 = WARNING, 1 = INFO, 2+ = DEBUG

    Returns
    -------
    logging.Logger
    """
    logger = logging.getLogger("csv2gpkg")
    if logger.handlers:
        for h in list(logger.handlers):
            logger.removeHandler(h)
    level = logging.WARNING if verbosity <= 0 else logging.INFO if verbosity == 1 else logging.DEBUG
    logger.setLevel(level)
    ch = logging.StreamHandler(stream=sys.stderr)
    ch.setLevel(level)
    fmt = logging.Formatter("[%(asctime)s] %(levelname)s - %(message)s", "%H:%M:%S")
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    return logger

LOGGER = setup_logger(VERBOSITY)


# ============================ Utility functions ============================== #
def infer_fiona_type(pd_dtype: str) -> str:
    """
    Map a pandas dtype string to a Fiona schema field type.

    Parameters
    ----------
    pd_dtype : str
        pandas dtype string (e.g. 'int64', 'float64', 'object', 'bool')

    Returns
    -------
    str
        Fiona type string (e.g. 'int', 'float', 'str', 'bool').
    """
    dt = str(pd_dtype)
    if "int" in dt:
        return "int"
    if "float" in dt:
        return "float"
    if "bool" in dt:
        return "bool"
    return "str"  # object/category/datetime -> store as string


def build_schema_from_df(df: pd.DataFrame, geometry_type: str = GEOM_SCHEMA_TYPE) -> Dict:
    """
    Build a Fiona schema from a DataFrame, excluding the 'geometry' column.

    Parameters
    ----------
    df : pd.DataFrame
        First chunk to infer attribute types.
    geometry_type : str
        Geometry type for the layer schema ("MultiPolygon" by default).

    Returns
    -------
    dict
        Fiona schema dict with declared geometry type and attribute field types.
    """
    properties = {}
    for col, dtype in df.dtypes.items():
        if col == "geometry":
            continue
        properties[col] = infer_fiona_type(str(dtype))
    return {"geometry": geometry_type, "properties": properties}


def _fast_parse_wkts(series: pd.Series):
    """
    Vectorized WKT -> shapely geometry if available (Shapely ≥2.0),
    else per-row fallback using shapely.wkt.loads.
    """
    if SHAPELY_HAS_FROM_WKT:
        return shapely_from_wkt(series.to_numpy(copy=False))
    return series.map(shapely_wkt.loads)


def repair_geometry(geom):
    """
    Attempt to repair invalid geometries:
      - Prefer Shapely make_valid (>=2.0).
      - Fallback to buffer(0) for minor self-intersections.
    Returns a valid geometry or None.
    """
    if geom is None:
        return None
    if geom.is_valid:
        return geom
    try:
        if REPAIR_INVALID and SHAPELY_HAS_MAKE_VALID:
            fixed = shapely_make_valid(geom)
        else:
            fixed = geom.buffer(0) if REPAIR_INVALID else geom
        if fixed is not None and not fixed.is_empty:
            return fixed
    except Exception:
        pass
    return None


def _as_multipolygon(geom):
    """
    Ensure the geometry is a MultiPolygon to match the layer schema.
    Returns None if geometry is missing/empty or of an unexpected type.
    """
    if geom is None or getattr(geom, "is_empty", True):
        return None
    gt = geom.geom_type
    if gt == "MultiPolygon":
        return geom
    if gt == "Polygon":
        return MultiPolygon([geom])
    # Unexpected geometry types (Point/LineString/etc.) are skipped
    return None


def iter_csv_chunks(path: str, chunk_size: int, usecols: Optional[List[str]] = None) -> Iterable[pd.DataFrame]:
    """
    Yield chunks of a CSV/CSV.GZ using pandas to keep memory bounded.
    """
    return pd.read_csv(
        path,
        compression="infer",
        chunksize=chunk_size,
        usecols=usecols,
        low_memory=True,
    )


def gpkg_feature_count(path: str, layer: str) -> Optional[int]:
    """
    Return feature count of a GPKG layer, or None if not readable.
    """
    try:
        with fiona.open(path, layer=layer) as src:
            return len(src)
    except Exception:
        return None


def csv_row_count_quick(path: str, geom_col: str, chunk_size: int = 500_000) -> int:
    """
    Quick-ish row count via pandas (reads only one column; supports gzip).
    """
    total = 0
    for df in pd.read_csv(path, compression="infer", chunksize=chunk_size, usecols=[geom_col], low_memory=True):
        total += len(df)
    return total


def open_fiona_sink(
    output_path: str,
    schema: Dict,
    crs_epsg: int,
    driver: str,
    layer: str = "buildings",
    overwrite: bool = False
):
    """
    Open a Fiona collection for writing (GPKG or GeoJSON).

    - For GPKG, fix geometry column name (GEOMETRY_NAME=geom) to make CreateSpatialIndex SQL deterministic.
    - Optionally disable spatial index during write for speed & to avoid rtree UNIQUE errors; we'll build it after.
    """
    if overwrite and os.path.exists(output_path):
        os.remove(output_path)

    crs = CRS.from_epsg(crs_epsg)

    if driver.upper() == "GPKG":
        layer_options = [f"GEOMETRY_NAME={GEOM_NAME}"]
        if not WRITE_WITH_SPATIAL_INDEX:
            layer_options.append("SPATIAL_INDEX=NO")
        if FID64:
            layer_options.append("FID64=YES")
        return fiona.open(
            output_path,
            mode="w",
            driver="GPKG",
            schema=schema,
            crs=crs,
            layer=layer,
            layer_options=layer_options
        )
    elif driver.upper() == "GEOJSON":
        return fiona.open(
            output_path,
            mode="w",
            driver="GeoJSON",
            schema=schema,
            crs=crs,
        )
    else:
        raise ValueError(f"Unsupported driver: {driver}")


def _props_records(df: pd.DataFrame, geom_field: str) -> List[dict]:
    """
    Convert attributes to list[dict] with NaNs -> None for OGR compatibility.
    """
    props_df = df.drop(columns=[geom_field])
    records = props_df.to_dict(orient="records")
    for rec in records:
        for k, v in rec.items():
            if pd.isna(v):
                rec[k] = None
    return records


# ----------------------- Spatial index helpers (robust) ---------------------- #
def _rtree_exists_sqlite(path: str, layer: str, geom_name: str) -> bool:
    """
    Check via sqlite3 whether the GeoPackage R*Tree for (layer, geom_name) exists.
    """
    try:
        tbl = f"rtree_{layer}_{geom_name}_rowid"
        with sqlite3.connect(path) as con:
            cur = con.execute(
                "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
                (tbl,)
            )
            return cur.fetchone() is not None
    except Exception:
        return False


def _build_spatial_index_inprocess_gdal(path: str, layer: str, geom_name: str) -> bool:
    """
    Build spatial index using GDAL Python bindings.
    Returns True on success, False otherwise.
    """
    if not HAS_GDAL_PY:
        return False
    try:
        ds = ogr.Open(path, update=1)
        if ds is None:
            return False
        sql = f"SELECT CreateSpatialIndex('{layer}','{geom_name}')"
        res = ds.ExecuteSQL(sql)
        if res is not None and hasattr(ds, "ReleaseResultSet"):
            ds.ReleaseResultSet(res)
        ds = None
        return True
    except Exception as e:
        LOGGER.debug(f"GDAL Python CreateSpatialIndex failed: {e}")
        return False


def _build_spatial_index_via_cli(path: str, layer: str, geom_name: str) -> bool:
    """
    Build spatial index using ogrinfo CLI as fallback.
    Returns True on success, False otherwise.
    """
    ogrinfo = shutil.which("ogrinfo")
    if ogrinfo is None:
        return False
    try:
        sql = f"SELECT CreateSpatialIndex('{layer}','{geom_name}')"
        cmd = [ogrinfo, path, "-sql", sql]
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if proc.returncode == 0:
            return True
        LOGGER.debug(f"ogrinfo stderr: {proc.stderr}")
        return False
    except Exception as e:
        LOGGER.debug(f"ogrinfo CreateSpatialIndex failed: {e}")
        return False


def build_spatial_index(path: str, layer: str, geom_name: str = "geom") -> None:
    """
    Ensure a GeoPackage spatial index exists by running CreateSpatialIndex.
    First detect if it already exists; if not, try GDAL Python, then ogrinfo CLI.
    """
    if not BUILD_SPATIAL_INDEX_AFTER:
        return
    if not os.path.exists(path):
        LOGGER.warning(f"Cannot build spatial index; file not found: {path}")
        return

    # If already present, report and exit quietly
    if _rtree_exists_sqlite(path, layer, geom_name):
        LOGGER.info(f"Spatial index already present for '{layer}' in {os.path.basename(path)}.")
        return

    ok = _build_spatial_index_inprocess_gdal(path, layer, geom_name)
    if not ok:
        ok = _build_spatial_index_via_cli(path, layer, geom_name)

    # Re-check if it now exists
    if ok or _rtree_exists_sqlite(path, layer, geom_name):
        LOGGER.info(f"Spatial index created for layer '{layer}' in {os.path.basename(path)}.")
    else:
        LOGGER.warning(f"Could not create spatial index for '{layer}' in {os.path.basename(path)} "
                       f"(GDAL Python/ogrinfo unavailable?).")


# ---------------------------- Writer (with backoff) -------------------------- #
def _write_with_backoff(sink, features: List[dict], initial_batch: int) -> None:
    """
    Write features to a Fiona sink using batch 'writerecords' with adaptive backoff.
    If a large batch fails (TransactionError), the batch size is halved until it succeeds.
    Falls back to per-feature writes as a last resort.

    Parameters
    ----------
    sink : fiona.Collection in write mode
    features : list of GeoJSON-like feature dicts
    initial_batch : int
        Starting batch size (e.g., WRITE_BATCH_SIZE)
    """
    n = len(features)
    if n == 0:
        return

    start = max(1, int(initial_batch))
    i = 0
    while i < n:
        size = min(start, n - i)
        # Try progressively smaller batches upon failure
        while True:
            try:
                batch = features[i:i+size]
                sink.writerecords(batch)
                i += size
                break  # proceed to next segment
            except Exception as ex:
                # If batch too big for a single transaction, halve it
                if size <= 1:
                    # Last resort: per-feature write; if still fails, re-raise
                    try:
                        sink.write(features[i])
                        i += 1
                        break
                    except Exception as ex2:
                        raise ex2
                size = size // 2  # back off and retry


def write_csv_to_vector(
    input_csv: str,
    output_path: str,
    driver: str = "GPKG",
    layer: str = "buildings",
    chunk_size: int = 100_000,
    crs_epsg: int = 4326,
    overwrite: bool = False,
    expected_geom_col: str = "geometry",
    selected_columns: Optional[List[str]] = None,
) -> Dict[str, int]:
    """
    Convert one CSV (or .csv.gz) into a vector file (GPKG/GeoJSON) by streaming in chunks.

    Returns
    -------
    dict with counts: {"rows_in": X, "features_out": Y, "skipped": Z}
    """
    LOGGER.info(f"Processing: {os.path.basename(input_csv)} -> {os.path.basename(output_path)}")

    # Ensure geometry column is included
    usecols = None
    if selected_columns is not None:
        cols = list(selected_columns)
        if expected_geom_col not in cols:
            cols.append(expected_geom_col)
        usecols = cols

    # Prime iterator for schema inference
    chunk_iter = iter_csv_chunks(input_csv, chunk_size=chunk_size, usecols=usecols)
    try:
        first_chunk = next(chunk_iter)
    except StopIteration:
        LOGGER.warning(f"No data found in {input_csv}")
        return {"rows_in": 0, "features_out": 0, "skipped": 0}

    if expected_geom_col not in first_chunk.columns:
        raise ValueError(f"Geometry column '{expected_geom_col}' not found in {input_csv}")

    # Declare schema as MultiPolygon (Polygons will be promoted)
    schema = build_schema_from_df(first_chunk, geometry_type=GEOM_SCHEMA_TYPE)

    rows_in = 0
    features_out = 0
    skipped = 0

    with open_fiona_sink(
        output_path=output_path,
        schema=schema,
        crs_epsg=crs_epsg,
        driver=driver,
        layer=layer,
        overwrite=True,  # always clean write when we commit to processing
    ) as sink:

        def process_df(df: pd.DataFrame):
            """
            Process a dataframe chunk using fast vectorized WKT parsing and batched writing
            with adaptive backoff. Repairs invalid geometries only where needed, then casts
            Polygon→MultiPolygon.
            """
            nonlocal rows_in, features_out, skipped
            rows_in += len(df)

            geoms = _fast_parse_wkts(df[expected_geom_col])

            # Repair only invalid geometries
            if REPAIR_INVALID:
                try:
                    invalid_mask = ~geoms.is_valid
                except Exception:
                    invalid_mask = [not g.is_valid for g in geoms]
                if getattr(invalid_mask, "any", lambda: any(invalid_mask))():
                    for i, is_bad in enumerate(invalid_mask):
                        if is_bad:
                            geoms[i] = repair_geometry(geoms[i])

            # Convert to features (skip None/empty; cast to MultiPolygon)
            props_list = _props_records(df, expected_geom_col)
            features = []
            for geom, props in zip(geoms, props_list):
                geom = _as_multipolygon(geom)
                if geom is None:
                    skipped += 1
                    continue
                try:
                    features.append({"type": "Feature", "properties": props, "geometry": mapping(geom)})
                except Exception:
                    skipped += 1

            if features:
                _write_with_backoff(sink, features, WRITE_BATCH_SIZE)
                features_out += len(features)

        # First + remaining chunks
        process_df(first_chunk)
        for df in chunk_iter:
            process_df(df)

    # Build spatial index right after closing the sink (GeoPackage only)
    if driver.upper() == "GPKG" and BUILD_SPATIAL_INDEX_AFTER:
        build_spatial_index(output_path, layer, GEOM_NAME)

    LOGGER.info(f"Done: rows_in={rows_in:,}, features_out={features_out:,}, skipped={skipped:,}")
    return {"rows_in": rows_in, "features_out": features_out, "skipped": skipped}


def discover_inputs(input_dir: str, pattern: str) -> List[str]:
    """Find input CSV files using a glob pattern."""
    return sorted(glob(os.path.join(input_dir, pattern)))


def derive_output_path(input_path: str, output_dir: str, fmt: str) -> str:
    """
    Compute the output path for a given input (handles .csv.gz).
    """
    base = os.path.splitext(os.path.basename(input_path))[0]
    if base.lower().endswith(".csv"):
        base = base[:-4]
    ext = ".gpkg" if fmt.lower() == "gpkg" else ".geojson"
    return os.path.join(output_dir, f"{base}{ext}")


def should_process_file(
    input_csv: str,
    output_path: str,
    layer: str,
    behavior: str = "auto",
    geom_col: str = "geometry",
) -> Tuple[bool, str]:
    """
    Decide whether to process this file based on EXISTING_BEHAVIOR.
    Returns (should_process, decision_reason).
    """
    if not os.path.exists(output_path):
        return True, "no_existing_output"

    if behavior == "replace":
        return True, "replace_existing"
    if behavior == "skip":
        return False, "skip_existing"
    if behavior == "abort":
        raise RuntimeError(f"Output already exists: {output_path}. EXISTING_BEHAVIOR=abort.")

    # behavior == "auto"
    try:
        out_count = gpkg_feature_count(output_path, layer)
        in_rows = csv_row_count_quick(input_csv, geom_col)
        if out_count is not None and in_rows is not None:
            if out_count >= max(0, in_rows - ALLOW_SKIPS):
                return False, f"auto_skip_complete (out={out_count}, in={in_rows})"
            else:
                return True, f"auto_replace_incomplete (out={out_count}, in={in_rows})"
    except Exception:
        return True, "auto_replace_fallback"
    return True, "auto_process_default"


def run_batch(
    input_dir: str,
    output_dir: str,
    pattern: str = "*.csv.gz",
    output_format: str = "gpkg",
    layer: str = "buildings",
    chunk_size: int = 100_000,
    crs_epsg: int = 4326,
    geom_col: str = "geometry",
    selected_columns: Optional[List[str]] = None,
) -> Dict[str, int]:
    """
    Discover inputs and convert each to its own output vector file.
    Also triggers in-process spatial index creation for GeoPackage outputs.
    """
    os.makedirs(output_dir, exist_ok=True)
    inputs = discover_inputs(input_dir, pattern)
    if not inputs:
        LOGGER.error("No input files found. Check INPUT_DIR and PATTERN.")
        return {"rows_in": 0, "features_out": 0, "skipped": 0}

    driver = "GPKG" if output_format.lower() == "gpkg" else "GeoJSON"

    total_in = total_out = total_skip = 0
    for in_csv in inputs:
        out_path = derive_output_path(in_csv, output_dir, output_format)

        try:
            process, reason = should_process_file(
                input_csv=in_csv,
                output_path=out_path,
                layer=layer,
                behavior=EXISTING_BEHAVIOR,
                geom_col=geom_col,
            )
        except RuntimeError as e:
            LOGGER.error(str(e))
            break

        if not process:
            LOGGER.info(f"SKIP {os.path.basename(out_path)} [{reason}]")
            continue
        else:
            LOGGER.info(f"PROCESS {os.path.basename(out_path)} [{reason}]")

        try:
            stats = write_csv_to_vector(
                input_csv=in_csv,
                output_path=out_path,
                driver=driver,
                layer=layer,
                chunk_size=chunk_size,
                crs_epsg=crs_epsg,
                overwrite=True,
                expected_geom_col=geom_col,
                selected_columns=selected_columns,
            )
            total_in  += stats["rows_in"]
            total_out += stats["features_out"]
            total_skip+= stats["skipped"]
        except Exception as ex:
            LOGGER.exception(f"Failed to process {in_csv}: {ex}")
            # Continue to next file

    LOGGER.info(f"ALL DONE. rows_in={total_in:,}, features_out={total_out:,}, skipped={total_skip:,}")
    return {"rows_in": total_in, "features_out": total_out, "skipped": total_skip}


# ================================ Execute =================================== #
os.makedirs(OUTPUT_DIR, exist_ok=True)
_ = run_batch(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    pattern=PATTERN,
    output_format=OUTPUT_FORMAT,
    layer=GPKG_LAYER,
    chunk_size=CHUNK_SIZE,
    crs_epsg=CRS_EPSG,
    geom_col=GEOM_COL,
    selected_columns=SELECTED_COLUMNS,
)


[19:40:49] INFO - PROCESS ANDAMAN___NICOBAR.gpkg [no_existing_output]
[19:40:49] INFO - Processing: ANDAMAN___NICOBAR.csv.gz -> ANDAMAN___NICOBAR.gpkg
[19:41:07] WARNING - Could not create spatial index for 'buildings' in ANDAMAN___NICOBAR.gpkg (GDAL Python/ogrinfo unavailable?).
[19:41:07] INFO - Done: rows_in=65,864, features_out=65,864, skipped=0
[19:41:54] INFO - PROCESS ANDHRA_PRADESH.gpkg [auto_process_default]
[19:41:54] INFO - Processing: ANDHRA_PRADESH.csv.gz -> ANDHRA_PRADESH.gpkg
[20:22:46] WARNING - Could not create spatial index for 'buildings' in ANDHRA_PRADESH.gpkg (GDAL Python/ogrinfo unavailable?).
[20:22:46] INFO - Done: rows_in=6,691,174, features_out=6,691,174, skipped=0
[20:22:46] INFO - PROCESS ARUNACHAL_PRADESH.gpkg [no_existing_output]
[20:22:46] INFO - Processing: ARUNACHAL_PRADESH.csv.gz -> ARUNACHAL_PRADESH.gpkg
[20:23:39] WARNING - Could not create spatial index for 'buildings' in ARUNACHAL_PRADESH.gpkg (GDAL Python/ogrinfo unavailable?).
[20:23:39] INFO - D